# 3.8: L2 penalty vs. decoupled decay

The practical upshot:

- With Adam-family optimizers, use **AdamW** (decoupled decay), not Adam with an L2 term. In PyTorch, `torch.optim.Adam(..., weight_decay=λ)` implements the L2 version, while `torch.optim.AdamW(..., weight_decay=λ)` implements decoupled decay. The argument has the same name but a different meaning.
- Common practice excludes **biases and normalization gains** ($\boldsymbol{\gamma}$, $\boldsymbol{\beta}$) from weight decay. These parameters are few, and shrinking a normalization gain toward zero changes the scale of the whole layer rather than simplifying the function. Embeddings are handled differently across codebases.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From Warmup plus decay for LLM pretraining
import math

def lr_at(step, peak_lr, warmup_steps, total_steps, min_lr_ratio=0.1):
    min_lr = peak_lr * min_lr_ratio
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(progress, 1.0)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

# PyTorch: apply via LambdaLR, which multiplies the optimizer's base lr
import torch

model = torch.nn.Linear(10, 10)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda step: lr_at(step, 1.0, warmup_steps=1000, total_steps=100_000))


In [ ]:
decay, no_decay = [], []
for name, p in model.named_parameters():
    if p.ndim < 2 or "norm" in name.lower():   # biases, LayerNorm/RMSNorm gains
        no_decay.append(p)
    else:
        decay.append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.1},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=3e-4,
)
